# Brain tumor detection: baseline YOLO11n in Google Colab

This notebook fine-tunes a small COCO-pretrained YOLO model on the `subset_v1` split prepared in `notebook.ipynb` (Step 3) and saves everything needed to reproduce and inspect the run.

**Before running**

1. Upload `data/subset_v1.zip` (≈33 MB) to Google Drive as `MyDrive/brain_tumor/subset_v1.zip`.
2. *Runtime → Change runtime type → T4 GPU*.
3. *Runtime → Run all*. Allow access to Google Drive when asked.

**Outputs** (in `MyDrive/brain_tumor/runs/<run name>/`): `args.yaml` (full training configuration), `run_info.json` (versions, settings, training time, best epoch, validation metrics), `results.csv` and `train_log.txt` (per-epoch logs), `learning_curves.png`, `results.png`, PR/F1 curves and confusion matrices, `weights/best.pt` (checkpoint selected on the validation split) and `weights/last.pt`, plus a zip of the whole run folder.

The **test split is not used** here. It is reserved for the final evaluation of the run chosen on validation results.

## 1. Environment

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "No NVIDIA GPU found"


zsh:1: command not found: nvidia-smi
No NVIDIA GPU found


In [3]:
%pip install ultralytics==8.4.190

  Using cached ultralytics-8.4.190-py3-none-any.whl.metadata (47 kB)
  Using cached opencv_python-5.0.0.93-cp37-abi3-macosx_13_0_arm64.whl.metadata (19 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached polars-2.1.0-py3-none-any.whl.metadata (11 kB)
  Using cached nvidia_ml_py-13.615.71-py3-none-any.whl.metadata (9.7 kB)
  Using cached ultralytics_thop-2.2.5-py3-none-any.whl.metadata (14 kB)
  Using cached ultralytics_platform-0.1.111-py3-none-any.whl.metadata (18 kB)
  Using cached polars_runtime_32-2.1.0-cp311-abi3-macosx_11_0_arm64.whl.metadata (1.5 kB)
  Using cached charset_normalizer-3.5.2-cp314-cp314-macosx_10_15_universal2.whl.metadata (46 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached anyio-4.16.0-py3-none-an

In [ ]:
import json
import logging
import platform
import shutil
import subprocess
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
from IPython.display import Image as IPImage, display
from PIL import Image
from ultralytics import YOLO
from ultralytics.utils import LOGGER

VERSIONS = {
    "python": platform.python_version(),
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(VERSIONS, indent=2))
assert ultralytics.__version__ == "8.4.190", "restart the runtime after installation (Runtime -> Restart session)"
if not torch.cuda.is_available():
    print("WARNING: no GPU. Select Runtime -> Change runtime type -> T4 GPU, otherwise training is very slow.")

## 2. Data

The zip contains the `subset_v1` folder in the Ultralytics layout (`images/{train,val,test}`, `labels/{train,val,test}`), `data.yaml`, `split_config.json` (method, seed and counts) and `split_manifest.csv` (source of every image). The dataset is unpacked to the Colab disk, which is faster than reading from Drive, and the file counts are checked against `split_config.json`.

In [ ]:
# Paths can be overridden with environment variables, e.g. for a local run
PROJECT_DIR = Path(os.environ.get("BT_PROJECT_DIR", "/content/drive/MyDrive/brain_tumor"))
WORK_DIR = Path(os.environ.get("BT_WORK_DIR", "/content"))
if IN_COLAB and str(PROJECT_DIR).startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DATA_ZIP = PROJECT_DIR / "subset_v1.zip"
DATA_DIR = WORK_DIR / "datasets" / "subset_v1"
RUNS_DIR = PROJECT_DIR / "runs"
assert DATA_ZIP.exists(), f"Upload subset_v1.zip to {DATA_ZIP}"

if not DATA_DIR.exists():
    with zipfile.ZipFile(DATA_ZIP) as zf:
        zf.extractall(DATA_DIR.parent)

split_config = json.loads((DATA_DIR / "split_config.json").read_text())
for split, expected in split_config["images_per_split"].items():
    n_images = sum(1 for p in (DATA_DIR / "images" / split).iterdir() if p.is_file())
    n_labels = len(list((DATA_DIR / "labels" / split).glob("*.txt")))
    assert n_images == n_labels == expected, (split, n_images, n_labels, expected)

# Ultralytics resolves a relative 'path' against its own datasets folder, so write an absolute one
DATA_YAML = DATA_DIR / "data_colab.yaml"
DATA_YAML.write_text((DATA_DIR / "data.yaml").read_text().replace("path: .", f"path: {DATA_DIR}"))

print(f"Split seed: {split_config['random_seed']}; class names: {split_config['class_names']}")
display(pd.DataFrame(split_config["images_per_split_and_folder"]).rename_axis("images"))
display(pd.DataFrame(split_config["objects_per_split_and_class"]).rename_axis("objects"))

## 3. Baseline configuration

| Setting | Value | Reason |
|---|---|---|
| Framework | Ultralytics 8.4.190 | pinned so the run can be repeated |
| Model / initial weights | YOLO11n, `yolo11n.pt` (COCO-pretrained, ≈2.6 M parameters) | small model that trains quickly on a free T4 |
| Image size | 512 | most source images are 512 × 512, so no upscaling is needed |
| Epochs | 50 | enough for fine-tuning a pretrained detector on ≈950 images |
| Batch size | 16 | |
| Optimizer, learning rate | SGD, `lr0 = 0.01`, final `lr0 · lrf = 0.0001` (linear decay), 3 warm-up epochs | set explicitly: with `optimizer="auto"` Ultralytics picks the optimizer and learning rate itself and ignores `lr0` |
| Seed | 42, `deterministic=True` | same seed as the data split |
| Checkpoint selection | `best.pt` = epoch with the highest validation fitness (Ultralytics: 0.1·mAP@0.50 + 0.9·mAP@0.50:0.95) | chosen on validation only |

All other hyperparameters, including augmentation (mosaic, HSV, flips, scale, translate), are Ultralytics defaults and are written to `args.yaml`. GPU training is not bit-for-bit deterministic even with a fixed seed, so a repeated run can differ slightly.

For the controlled experiment, copy `BASELINE`, change exactly one setting and give the run a new `name`.

In [ ]:
SMOKE_TEST = os.environ.get("BT_SMOKE_TEST") == "1"   # tiny CPU run that only checks the notebook works

BASELINE = {
    "model": "yolo11n.pt",
    "data": str(DATA_YAML),
    "imgsz": 512,
    "epochs": 50,
    "batch": 16,
    "optimizer": "SGD",
    "lr0": 0.01,
    "lrf": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 3.0,
    "cos_lr": False,
    "patience": 50,          # no early stopping within 50 epochs
    "seed": 42,
    "deterministic": True,
    "workers": 2,
    "plots": True,
    "project": str(RUNS_DIR),
    "name": "baseline_yolo11n_512",
    "exist_ok": False,       # a repeated run gets a new folder (..._5122) instead of overwriting
}
if SMOKE_TEST:
    BASELINE.update(epochs=1, imgsz=256, batch=8, fraction=0.1, device="cpu", workers=0,
                    name="smoke_test", exist_ok=True)
print(json.dumps(BASELINE, indent=2))

## 4. Training

In [ ]:
log_path = WORK_DIR / "train_log.txt"
log_handler = logging.FileHandler(log_path, mode="w")
log_handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
LOGGER.addHandler(log_handler)

model = YOLO(BASELINE["model"])
started = datetime.now(timezone.utc)
t0 = time.perf_counter()
try:
    model.train(**BASELINE)
finally:
    LOGGER.removeHandler(log_handler)
    log_handler.close()
train_seconds = time.perf_counter() - t0

run_dir = Path(model.trainer.save_dir)
shutil.copy2(log_path, run_dir / "train_log.txt")
print(f"Run folder: {run_dir}\nTraining time: {train_seconds / 60:.1f} min")

## 5. Best checkpoint on the validation split

`best.pt` is evaluated again on the validation split with the standard Ultralytics settings for mAP (`conf=0.001`, NMS `iou=0.7`). Precision and recall are reported at the confidence threshold that maximises the mean F1 score over classes. The settings, versions, training time and metrics are written to `run_info.json`.

In [ ]:
best_path = run_dir / "weights" / "best.pt"
VAL_SETTINGS = {"split": "val", "imgsz": BASELINE["imgsz"], "batch": BASELINE["batch"], "conf": 0.001, "iou": 0.7}

val = YOLO(best_path).val(data=str(DATA_YAML), plots=True, project=str(run_dir), name="val_best", exist_ok=True,
                          device=BASELINE.get("device"), **VAL_SETTINGS)
names = val.names
per_class = pd.DataFrame(
    [dict(zip(["precision", "recall", "mAP50", "mAP50-95"], val.box.class_result(i)), cls=names[int(c)])
     for i, c in enumerate(val.box.ap_class_index)]).set_index("cls")
per_class.loc["all"] = [val.box.mp, val.box.mr, val.box.map50, val.box.map]
display(per_class.round(3))

results = pd.read_csv(run_dir / "results.csv")
results.columns = results.columns.str.strip()
fitness = 0.1 * results["metrics/mAP50(B)"] + 0.9 * results["metrics/mAP50-95(B)"]
best_row = results.loc[fitness.idxmax()]

run_info = {
    "run_name": run_dir.name,
    "started_utc": started.strftime("%Y-%m-%dT%H:%M:%SZ"),
    "versions": VERSIONS,
    "dataset": {"version": split_config["dataset_version"], "split_seed": split_config["random_seed"],
                "images_per_split": split_config["images_per_split"], "class_names": split_config["class_names"]},
    "model": "YOLO11n",
    "initial_weights": BASELINE["model"],
    "settings": {k: v for k, v in BASELINE.items() if k not in ("data", "project")},
    "epochs_completed": int(results["epoch"].max()),
    "training_time_seconds": round(train_seconds, 1),
    "training_time_minutes": round(train_seconds / 60, 2),
    "best_epoch": int(best_row["epoch"]),
    "best_checkpoint": str(best_path.relative_to(run_dir)),
    "validation_settings": VAL_SETTINGS,
    "validation_metrics_best": {"precision": val.box.mp, "recall": val.box.mr,
                                "mAP50": val.box.map50, "mAP50-95": val.box.map},
    "validation_per_class": per_class.drop(index="all").to_dict(orient="index"),
}
(run_dir / "run_info.json").write_text(json.dumps(run_info, indent=2, default=float))
(run_dir / "environment.txt").write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"],
                                                        capture_output=True, text=True).stdout)
print(json.dumps({k: run_info[k] for k in ["best_epoch", "training_time_minutes", "validation_metrics_best"]},
                 indent=2, default=float))

## 6. Learning curves and validation plots

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, loss in zip(axes[:3], ["box_loss", "cls_loss", "dfl_loss"]):
    ax.plot(results["epoch"], results[f"train/{loss}"], label="train")
    ax.plot(results["epoch"], results[f"val/{loss}"], label="val")
    ax.set_title(loss)
    ax.set_xlabel("epoch")
    ax.legend()
for metric, label in [("metrics/precision(B)", "precision"), ("metrics/recall(B)", "recall"),
                      ("metrics/mAP50(B)", "mAP@0.50"), ("metrics/mAP50-95(B)", "mAP@0.50:0.95")]:
    axes[3].plot(results["epoch"], results[metric], label=label)
axes[3].axvline(run_info["best_epoch"], color="grey", ls="--", lw=1, label=f"best epoch {run_info['best_epoch']}")
axes[3].set_title("validation metrics")
axes[3].set_xlabel("epoch")
axes[3].set_ylim(0, 1)
axes[3].legend(fontsize=8)
fig.suptitle(run_dir.name)
plt.tight_layout()
fig.savefig(run_dir / "learning_curves.png", dpi=120)
plt.show()

val_dir = run_dir / "val_best"
for name in ["confusion_matrix_normalized.png", "BoxPR_curve.png", "BoxF1_curve.png"]:
    path = val_dir / name
    if path.exists():
        print(name)
        display(IPImage(filename=str(path), width=600))

## 7. Sanity check: predictions on validation images

A few validation images with predicted boxes (confidence ≥ 0.25) next to the ground truth. Test images are not shown here.

In [ ]:
COLORS = ["#ff3b30", "#ffcc00", "#0a84ff"]
manifest = pd.read_csv(DATA_DIR / "split_manifest.csv")
examples = (manifest[manifest["split"] == "val"].groupby("folder", group_keys=False)
            .apply(lambda g: g.sample(2, random_state=42)))
best_model = YOLO(best_path)

fig, axes = plt.subplots(2, len(examples) // 2, figsize=(4 * len(examples) // 2, 8.5))
for ax, rec in zip(axes.flat, examples.itertuples()):
    image_path = DATA_DIR / "images" / "val" / rec.file
    image = Image.open(image_path).convert("RGB")
    W, H = image.size
    ax.imshow(image)
    for line in filter(None, (DATA_DIR / "labels" / "val" / f"{image_path.stem}.txt").read_text().splitlines()):
        c, xc, yc, w, h = int(line.split()[0]), *map(float, line.split()[1:])
        ax.add_patch(plt.Rectangle(((xc - w / 2) * W, (yc - h / 2) * H), w * W, h * H, fill=False,
                                   ec="white", lw=1.5, ls="--"))
    pred = best_model.predict(image_path, imgsz=BASELINE["imgsz"], conf=0.25, verbose=False,
                              device=BASELINE.get("device"))[0]
    for (x0, y0, x1, y1), c, p in zip(pred.boxes.xyxy.tolist(), pred.boxes.cls.int().tolist(), pred.boxes.conf.tolist()):
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec=COLORS[c], lw=2))
        ax.text(x0, y0 - 3, f"{names[c]} {p:.2f}", fontsize=7, va="bottom", bbox=dict(fc=COLORS[c], ec="none", pad=1))
    ax.set_title(f"{rec.folder}: {rec.file}\nGT dashed white, prediction coloured", fontsize=7)
    ax.axis("off")
plt.tight_layout()
fig.savefig(run_dir / "val_predictions.png", dpi=110)
plt.show()

## 8. Save the run

In [ ]:
archive = shutil.make_archive(str(RUNS_DIR / run_dir.name), "zip", root_dir=run_dir.parent, base_dir=run_dir.name)
print("Run folder :", run_dir)
print("Best weights:", best_path, f"({best_path.stat().st_size / 2**20:.1f} MB)")
print("Run archive:", archive)
for p in sorted(run_dir.iterdir()):
    print("  ", p.name)